In [1]:
import pandas as pd
import scipy
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import emoji
import re
import datetime


import pandas as pd
from transformers import pipeline
from tqdm import tqdm

from transformers import AutoConfig
from scipy.special import softmax

In [2]:
tweets = pd.read_csv("../Dataset/bitcoin-tweets-2021.csv", encoding='utf-8', lineterminator='\n', low_memory=False)
tweets.head()

,datetime,username,text
0,2021-01-01 23:59:58+00:00,WoodooProd,Crypto adoption in 2021: Top trends and predic...
1,2021-01-01 23:59:53+00:00,YSojourners,Ripple Executive Says US Laws on Crypto Are Va...
2,2021-01-01 23:59:53+00:00,Andama_n,@hextoysofficial | https://t.co/C6LPpuq05n Are...
3,2021-01-01 23:59:51+00:00,YSojourners,FUN and Games: Altcoin Drastically Pumps And D...
4,2021-01-01 23:59:50+00:00,HunnerSa,I bought #Bitcoin at 6k-8k-9k-12k-16k-19k and ...


In [3]:
def first_day(df):
    df = df[df['datetime'].str.contains('2021-02-10')]
    df['postdate'] = pd.to_datetime(df['datetime']).dt.date
    df['posttime'] = pd.to_datetime(df['datetime']).dt.time
    return df

firstday = first_day(tweets) # 60353 tweets
firsthour_tweets = firstday[firstday['posttime'] < datetime.time(1, 0 , 0)] # 1440 tweets

/tmp/ipykernel_6940/3151282841.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['postdate'] = pd.to_datetime(df['datetime']).dt.date
/tmp/ipykernel_6940/3151282841.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['posttime'] = pd.to_datetime(df['datetime']).dt.time


In [4]:
def preprocess_emoji_test(text):
    if pd.isna(text):
        return ""

    # convert emoji to text 
    text = emoji.demojize(text, language='en')

    # remove urls 
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # remove spaces
    text = " ".join(text.split())

    # lowercase
    text = text.lower()

    return text

firsthour_tweets["c_text"] = firsthour_tweets["text"].apply(preprocess_emoji_test) 

/tmp/ipykernel_6940/1566022966.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  firsthour_tweets["c_text"] = firsthour_tweets["text"].apply(preprocess_emoji_test)


In [5]:
btc = pd.read_csv("../Dataset/btcusd_1-min_data.csv")
btc['datetime'] = pd.to_datetime(btc['Timestamp'],unit='s')
btc['date'] = pd.to_datetime(btc['datetime']).dt.date
btc['time'] = pd.to_datetime(btc['datetime']).dt.time

In [6]:
sameday_btc = btc[btc['date'] == datetime.date(2021, 2, 10)]
sameday_btc

,Timestamp,Open,High,Low,Close,Volume,datetime,date,time
4791719,1.612915e+09,47072.73,47094.48,47031.93,47070.92,2.580461,2021-02-10 00:00:00,2021-02-10,00:00:00
4791720,1.612915e+09,47069.71,47133.54,47069.71,47104.45,3.023258,2021-02-10 00:01:00,2021-02-10,00:01:00
4791721,1.612915e+09,47113.37,47200.00,47113.37,47156.75,13.636693,2021-02-10 00:02:00,2021-02-10,00:02:00
4791722,1.612915e+09,47158.75,47232.64,47158.75,47187.92,6.975702,2021-02-10 00:03:00,2021-02-10,00:03:00
4791723,1.612915e+09,47169.17,47216.23,47155.46,47189.08,4.717993,2021-02-10 00:04:00,2021-02-10,00:04:00
...,...,...,...,...,...,...,...,...,...
4793154,1.613001e+09,45078.64,45105.38,45074.16,45105.38,2.610335,2021-02-10 23:55:00,2021-02-10,23:55:00
4793155,1.613001e+09,45105.38,45105.38,45015.71,45054.94,8.961133,2021-02-10 23:56:00,2021-02-10,23:56:00
4793156,1.613001e+09,45037.12,45066.12,45000.00,45066.12,1.141000,2021-02-10 23:57:00,2021-02-10,23:57:00
4793157,1.613001e+09,45052.42,45199.16,45033.11,45199.16,3.369465,2021-02-10 23:58:00,2021-02-10,23:58:00


In [ ]:
firsthour_tweets['text'].iloc[0] # text without preprocessing  

'https://t.co/eJMhX206gp\n\nCashaa in the conversation along side Bitcoin and Etherium. Skip to 2min 30seconds. \n\nIt’s not too late to get onboard the Cashaa 🚂 Toot Toot\n\nCashaa #CAS #Cashaa #cryptoinvesting #bitcoin #ethereum'

In [ ]:
firsthour_tweets['c_text'].iloc[0] # text with preprocesssing


'cashaa in the conversation along side bitcoin and etherium. skip to 2min 30seconds. it’s not too late to get onboard the cashaa :locomotive: toot toot cashaa #cas #cashaa #cryptoinvesting #bitcoin #ethereum'

In [ ]:
robertaMODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
robertatokenizer = AutoTokenizer.from_pretrained(robertaMODEL)
config = AutoConfig.from_pretrained(robertaMODEL)
# PT
robertamodel = AutoModelForSequenceClassification.from_pretrained(robertaMODEL)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
## testing first line with un processed text for evaluation
#  
# text = "https://t.co/eJMhX206gp\n\nCashaa in the conversation along side Bitcoin and Etherium. Skip to 2min 30seconds. \n\nIt’s not too late to get onboard the Cashaa 🚂 Toot Toot\n\nCashaa #CAS #Cashaa #cryptoinvesting #bitcoin #ethereum"

# encoded_input = robertatokenizer(text, return_tensors='pt')
# output = robertamodel(**encoded_input)
# scores = output[0][0].detach().numpy()
# scores = softmax(scores)

# ranking = np.argsort(scores)
# ranking = ranking[::-1]
# for i in range(scores.shape[0]):
#     l = config.id2label[ranking[i]]
#     s = scores[ranking[i]]
#     print(f"{i+1}) {l} {np.round(float(s), 4)}")

1) neutral 0.5169
2) positive 0.4784
3) negative 0.0047


In [ ]:
def robert_test(data):
    X = data['c_text'].iloc[:].to_list()
    
    preds = []
    preds_proba = []
    for x in X:
        # with torch.no_grad():
        encoded_input = robertatokenizer(x, return_tensors='pt')
        logits = robertamodel(**encoded_input).logits
        output = robertamodel(**encoded_input)
        # input_sequence = tokenizer(x, return_tensors="pt", **tokenizer_kwargs)
        # logits = model(**input_sequence).logits
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        ranking = np.argsort(scores)
        ranking = ranking[::-1]
        sent_pred = config.id2label[ranking[0]]
        # print(x, sent_pred)

        preds.append(sent_pred)
        preds_proba.append(scores[ranking[0]])

    data['sentiment'] = preds
    data['sentiment_score'] = preds_proba

    return data


In [ ]:
## test for ten tweets with the function
# testtentwts = robert_test(firsthour_tweets) 

cashaa in the conversation along side bitcoin and etherium. skip to 2min 30seconds. it’s not too late to get onboard the cashaa :locomotive: toot toot cashaa #cas #cashaa #cryptoinvesting #bitcoin #ethereum positive
@jakeb515 it's just a really bad time to get them rn, covid delayed the availability of parts and people are buying all the high-end ones to farm bitcoin. negative
@bernieshq where are you getting your disinformation??? if you still think i am wrong, please send me links that support your position - on bitcoin specifically, not the underlying blockchain technology. negative
bitcoin seems like a pretty big waste of energy negative
we're gonna need a bigger moon #wbitg1tga #bitcoin neutral
i earned with the help of my friends! :money_with_wings: anyone can try how it is simple and fun on @betfury_io i-gaming platform. my username: antho1978 #betfurysuccess #tron #bitcoin #referral positive
i still cannot believe #bitcoin pumped so hard because a discord trading server started

In [46]:
res2 = robert_test(firsthour_tweets)

/tmp/ipykernel_6940/18919443.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentiment'] = preds
/tmp/ipykernel_6940/18919443.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sentiment_score'] = preds_proba


In [ ]:
res2.head()

,datetime,username,text,postdate,posttime,c_text,sentiment,sentiment_score
1882557,2021-02-10 00:59:59+00:00,mickboylan,https://t.co/eJMhX206gp\n\nCashaa in the conve...,2021-02-10,00:59:59,cashaa in the conversation along side bitcoin ...,positive,0.642417
1882558,2021-02-10 00:59:58+00:00,Brett_Beech,@jakeb515 it's just a really bad time to get t...,2021-02-10,00:59:58,@jakeb515 it's just a really bad time to get t...,negative,0.920326
1882559,2021-02-10 00:59:57+00:00,AmerLiberal,@bernieshq WHERE are you getting your disinfor...,2021-02-10,00:59:57,@bernieshq where are you getting your disinfor...,negative,0.843556
1882560,2021-02-10 00:59:56+00:00,DHusko23,Bitcoin seems like a pretty big waste of energy,2021-02-10,00:59:56,bitcoin seems like a pretty big waste of energy,negative,0.893014
1882561,2021-02-10 00:59:56+00:00,POSTkingPiNwin,We're Gonna Need A Bigger Moon\n#WBITG1TGA #Bi...,2021-02-10,00:59:56,we're gonna need a bigger moon #wbitg1tga #bit...,neutral,0.588151
